# R19-H211 - The last eight: render and carrier-linkage policy for the unranked residue

**Hypothesis** - (a) a carrier-linkage census identifies why each of the 8 present-but-unranked golds' carriers never seeds (rank position, embedding text, missing device link - enumerated per gold); (b) the cheapest sufficient policy mix (raising the per-product relation render cap and/or deterministic SKU->device linkage, simulated at render time) recovers >= 5/8 at unchanged k and token budget within the H171 knee (+5.60%), with zero regression on the pinned census's 77 hits.

**Harness** - the pinned H207 canonical harness verbatim (neo4j2 read-only, explicit driver; canonical render spec mirroring `pipeline._retrieve_local`; deterministic CPU scorer with the exact-identifier arm; render + graph fingerprints stamped and asserted against the pinned census; seeds sorted score-desc,id-asc). Baseline: recall@64 = 77/101 on `probes-wide-h188.json`; the 8 unranked misses (5 catalogue codes, 2 spec table cells, 1 spec sentence) from `reports/render-parity-h207-20260707T155614Z.json`.

**Constraints** - neo4j2 READ-ONLY; all policy effects simulated at render time (no graph writes); CPU-only.


In [1]:
# CPU-only; pin retrieval to neo4j2 via an explicit driver (NOT .env / Foundry defaults) - H207 harness verbatim
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""                     # CPU-only
import re, json, pickle, hashlib, itertools, unicodedata, datetime
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
from neo4j import GraphDatabase
from knowledge_graph_foundry import load_settings
from knowledge_graph_foundry.graph.graphrag import vector_query
from rich import print as rprint

ROOT = Path("..")
NEO4J2 = "bolt://user-konrad.jelen-kgf-neo4j2:7687"         # pinned baseline, READ-ONLY
AUTH = ("neo4j", os.environ.get("NEO4J_PASSWORD", "kgfoundry"))
settings = load_settings(ROOT / "config.yml")
VEC = settings.graphrag.vector_index_name
K64, RETRIEVE_TOPK, REL_LIMIT = 64, 128, 15
KNEE_PCT = 5.60                                              # H171 knee anchor (+5.60% tokens)
driver2 = GraphDatabase.driver(NEO4J2, auth=AUTH)           # every vector_query uses THIS driver
rprint(f"[cyan]config[/cyan] vec={VEC} eval_k={K64} retrieve_top_k={RETRIEVE_TOPK} rel_cap={REL_LIMIT} knee=+{KNEE_PCT}% (CPU-only, neo4j2 pinned, READ-ONLY)")

2026-07-07 21:07:08.789 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


config vec=kgf_entity_embeddings eval_k=64 retrieve_top_k=128 rel_cap=15 knee=+5.6% (CPU-only, neo4j2 pinned, 
READ-ONLY)

In [2]:
# Graph pull from neo4j2 (READ-ONLY) + production-faithful render primitives (H207 verbatim; rel cap parameterized for policy A)
with driver2.session() as s:
    ents = s.run("MATCH (e:Entity) RETURN e.id AS id, e.name AS name, e.description AS description, "
                 "properties(e) AS props, labels(e) AS types").data()
    edges = s.run("MATCH (a:Entity)-[r]-(b:Entity) WHERE type(r)<>'SIMILAR_TO' AND a.id<b.id "
                  "RETURN DISTINCT a.id AS a, b.id AS b, type(r) AS rel").data()
    prop_rows = s.run("MATCH (p:Proposition)-[:ABOUT]->(e:Entity) RETURN e.id AS eid, p.text AS text").data()
    alias_rows = s.run("MATCH (e:Entity)-[:SAME_AS*1..2]-(a:Entity) WHERE e.id<>a.id "
                       "RETURN e.id AS eid, collect(DISTINCT a.id)[..5] AS aliases").data()
    emb_head = {r["id"]: r["head"] for r in s.run(
        "MATCH (e:Entity) WHERE e.embedding IS NOT NULL RETURN e.id AS id, e.embedding[0..8] AS head").data()}
node = {r["id"]: r for r in ents}; names = {r["id"]: r["name"] for r in ents}
props_by = defaultdict(list); [props_by[r["eid"]].append(r["text"]) for r in prop_rows]
alias_by = {r["eid"]: r["aliases"] for r in alias_rows}
rels_by = defaultdict(list)
for e in edges:
    rels_by[e["a"]].append((e["rel"], e["b"])); rels_by[e["b"]].append((e["rel"], e["a"]))

def spec_of(r): return {k.removeprefix("prop_"): v for k, v in r["props"].items() if k.startswith("prop_")}
def merged_spec(nid):
    r = node[nid]; spec = dict(spec_of(r))
    for a in [a for a in alias_by.get(nid, []) if a in node]:
        for k, v in spec_of(node[a]).items(): spec.setdefault(k, v)
    return spec
def base_render(nid):
    r = node[nid]; spec = merged_spec(nid); al = [a for a in alias_by.get(nid, []) if a in node]
    aka = (f"Also known as: {', '.join(names.get(a, '') for a in al)}\n" if al else "")
    return f"## {r['name']} ({', '.join(r['types'])})\n{aka}{r['description'] or ''}\nProperties: {json.dumps(spec, default=str)}"
def seed_render(nid, cap=REL_LIMIT):
    rels = "; ".join(f"{t} -> {names.get(b, '')}" for t, b in rels_by.get(nid, [])[:cap])
    return base_render(nid) + "\nRelations: " + rels
def units_of(ids, cap=REL_LIMIT): return [seed_render(n, cap) for n in ids] + [t for n in ids for t in props_by.get(n, [])]
rprint(f"[green]pulled neo4j2[/green] entities {len(node)} edges {len(edges)} propositions {len(prop_rows)}")

pulled neo4j2 entities 2798 edges 3905 propositions 19654

In [3]:
# ---- deterministic CPU scorer (H207 verbatim: exact_present incl. the H206 exact-identifier behaviour U word-overlap) ----
_TM = dict.fromkeys(map(ord, "\u00ae\u2122\u00a9"), None)
def gnorm(s):
    s = (s or "").translate(_TM); s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00a0", " ").replace("\u00d7", "x").replace("*", "x").replace("\u00b7", "x")
    s = re.sub(r"(?<=\d),(?=\d)", "", s)
    return re.sub(r"\s+", " ", s.casefold()).strip()
GLYPH = {'\u2122':'', '\u00ae':'', '\u00a9':'', '\u2013':'-', '\u2014':'-', '\u00a0':' ', '\u2009':' ',
         '\u202f':' ', '\ufb01':'fi', '\ufb02':'fl', '\u2032':"'", '\u2033':'"', '\u00b0':' ', '\u00d7':'x'}
def gnorm2(s):
    s = s or ""
    for k, v in GLYPH.items(): s = s.replace(k, v)
    s = unicodedata.normalize("NFKD", s); s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).casefold().strip()
_UNIT = r"(cmh2o|cm h2o|mm|cm|dba|db\(a\)|db|kg|g|oz|ml|l|w|hz|watts?|mins?|hours?|years?|m)"
def _numunits(t): return re.findall(r"(\d[\d,\.]*)\s*" + _UNIT + r"?", gnorm2(t))
def exact_present(gold, ctx):
    g = gnorm2(gold); c = gnorm2(ctx)
    if g and g in c: return True
    gd = re.sub(r"[ ,]", "", g); cd = re.sub(r"[ ,]", "", c)
    if any(ch.isdigit() for ch in gd) and len(gd) >= 4 and gd in cd: return True
    gnu = _numunits(gold)
    if gnu:
        for num, unit in gnu:
            nd = num.replace(",", "")
            if unit:
                if not (re.search(r"(?<!\d)" + re.escape(nd) + r"\s*" + re.escape(unit), c) or
                        re.search(re.escape(nd) + re.escape(unit), cd)): return False
            else:
                if not re.search(r"(?<!\d)" + re.escape(nd) + r"(?!\d)", cd): return False
        return True
    return False
def word_overlap_ctx(gold, ctx, thr=0.6):
    ng = gnorm(gold); c = gnorm(ctx)
    w = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ng)); cw = set(re.findall(r"[a-z][a-z0-9\-]{2,}", c))
    return bool(w) and len(w & cw) / len(w) >= thr
def is_numeric_gold(g): return bool(re.search(r"\d", g))
def det_present(gold, ids, cap=REL_LIMIT, extra=""):
    ctx = " ".join(units_of(ids, cap)) + (" " + extra if extra else "")
    if is_numeric_gold(gold): return exact_present(gold, ctx)
    if exact_present(gold, ctx): return True
    return word_overlap_ctx(gold, ctx)
def carriers(gold):
    if is_numeric_gold(gold):
        return [nid for nid in node if exact_present(gold, seed_render(nid) + " " + " ".join(props_by.get(nid, [])))]
    return [nid for nid in node if word_overlap_ctx(gold, " ".join(units_of([nid]))) or exact_present(gold, seed_render(nid))]
TOK = lambda s: max(1, len(re.sub(r"\s+", " ", (s or "").casefold())) // 4)   # H171 token proxy
rprint("[green]scorer ready[/green] deterministic exact_present (numeric/code incl. H206 identifier behaviour) U word-overlap>=0.6; CPU-only")

scorer ready deterministic exact_present (numeric/code incl. H206 identifier behaviour) U word-overlap>=0.6; 
CPU-only

In [4]:
# ---- fingerprints stamped at start; MUST match the pinned H207 census ----
CANON_SPEC = dict(
    source="pipeline._retrieve_local entity_blocks (production query path)",
    per_seed=["## name (types)", "Also known as (SAME_AS*1..2, <=5)", "description",
              "Properties: json(prop_* keys, alias-merged)", "Relations: type -> name (<=15, non-SIMILAR_TO)"],
    proposition_channel="per-seed attached propositions (Proposition-[:ABOUT]->seed)",
    eval_k=K64, retrieve_top_k=RETRIEVE_TOPK, rel_limit=REL_LIMIT,
    seed_order="score desc, id asc (deterministic tie-break)",
    scorer="deterministic: exact_present(numeric/code) OR word_overlap>=0.6(prose); CPU-only, no NLI")
def render_fingerprint(spec, sample_ids):
    blob = json.dumps({k: spec[k] for k in sorted(spec)}, default=str) + "\x1e" + \
           "\x1e".join(seed_render(c) for c in sorted(sample_ids))
    return hashlib.sha256(blob.encode()).hexdigest()[:16]
def graph_fingerprint(carrier_ids):
    render_blob = "\x1e".join(seed_render(c) for c in sorted(carrier_ids))
    emb_blob = ";".join(f"{c}:" + ",".join(f"{x:.4f}" for x in emb_head.get(c, [])) for c in sorted(carrier_ids))
    return dict(node_count=len(node), edge_count=len(edges), embedding_count=len(emb_head),
                content_hash=hashlib.sha256(render_blob.encode()).hexdigest()[:16],
                embedding_digest=hashlib.sha256(emb_blob.encode()).hexdigest()[:16])
ALL_IDS = sorted(node)
GRAPH_FP = graph_fingerprint(ALL_IDS)
RENDER_FP = render_fingerprint(CANON_SPEC, ALL_IDS)
PINNED = dict(render_fp="96ab16d299fbbc71", content_hash="6fdc41bde495d1a3")
assert RENDER_FP == PINNED["render_fp"], f"render fingerprint drift: {RENDER_FP}"
assert GRAPH_FP["content_hash"] == PINNED["content_hash"], f"graph content drift: {GRAPH_FP['content_hash']}"
rprint(f"[magenta]render_fingerprint[/magenta] {RENDER_FP}  [magenta]graph_fingerprint[/magenta] {GRAPH_FP}  -> [green]MATCH pinned H207[/green]")

render_fingerprint 96ab16d299fbbc71  graph_fingerprint {'node_count': 2798, 'edge_count': 3905, 'embedding_count': 
2798, 'content_hash': '6fdc41bde495d1a3', 'embedding_digest': '2a3908456d2e2d8c'}  -> MATCH pinned H207

In [5]:
# ---- probes + retrieval (H207 verbatim) + pinned-census reproduction ----
W = json.load(open(ROOT / "data/processed/probes-wide-h188.json"))["probes"]
wgolds = [(p["id"], g, p) for p in W for g in p["gold_evidence"]]
qcache = pickle.load(open(".wide_probes_h188_qcache.pkl", "rb"))
def qkey(q): return hashlib.md5(q.encode()).hexdigest()
def seeds_from(p, topk=RETRIEVE_TOPK, k=K64):
    res = vector_query(driver2, qcache[qkey(p["question"])], VEC, top_k=topk)
    res = sorted(res, key=lambda x: (-x["score"], x["id"]))   # deterministic tie-break
    return [x["id"] for x in res if x["id"] in node][:k]
def full_rank(p, topk=512):
    res = vector_query(driver2, qcache[qkey(p["question"])], VEC, top_k=topk)
    res = sorted(res, key=lambda x: (-x["score"], x["id"]))
    order = [(x["id"], x["score"]) for x in res if x["id"] in node]
    return {nid: (i + 1, sc) for i, (nid, sc) in enumerate(order)}
TOP64 = {p["id"]: seeds_from(p) for p in W}
pby = {p["id"]: p for p in W}
BASE_HITS = {(pid, g) for pid, g, p in wgolds if det_present(g, TOP64[pid])}
rprint(f"[bold cyan]pinned census reproduction[/bold cyan] hits = {len(BASE_HITS)}/{len(wgolds)} "
       f"(recall@64 = {len(BASE_HITS)/len(wgolds):.3f})  -> [{'green' if len(BASE_HITS)==77 else 'red'}]"
       f"{'MATCH 77' if len(BASE_HITS)==77 else 'MISMATCH'}[/]")
EIGHT = ["W004", "W010", "W012", "W044", "W046", "W063", "W065", "W066"]   # the pinned unranked residue
assert all((pid, pby[pid]["gold_evidence"][0]) not in BASE_HITS for pid in EIGHT)
BASE_TOK = sum(TOK(seed_render(n)) + sum(TOK(t) for t in props_by.get(n, [])) for p in W for n in TOP64[p["id"]])
KNEE_BUDGET = int(BASE_TOK * KNEE_PCT / 100)
rprint(f"census render-token baseline = {BASE_TOK}  knee budget (+{KNEE_PCT}%) = {KNEE_BUDGET} tokens")

pinned census reproduction hits = 77/101 (recall@64 = 0.762)  -> MATCH 77

census render-token baseline = 2140060  knee budget (+5.6%) = 119843 tokens

## Task 1 - Per-gold forensics on the 8 unranked golds

For each gold: carriers located in neo4j2, true similarity rank of the best carrier against the probe query (top_k=512), the carrier's embedding surface (name + description - the text the vector index sees), and the cause classification.


In [6]:
# ---- per-gold forensics: carriers, true similarity ranks, embedding surface, cause ----
FORENSICS = []
for pid in EIGHT:
    p = pby[pid]; gold = p["gold_evidence"][0]
    rankmap = full_rank(p); cs = carriers(gold)
    ranked = sorted([(c, *rankmap.get(c, (9999, None))) for c in cs], key=lambda x: x[1])
    best_c, best_rank, best_score = ranked[0]
    FORENSICS.append(dict(
        pid=pid, gold=gold, rule=p["derivation_rule"], question=p["question"],
        n_carriers=len(cs),
        carriers=[dict(id=c, name=names[c], rank=(r if r < 9999 else ">512"),
                       score=(round(sc, 3) if sc else None), types=[t for t in node[c]["types"] if t != "Entity"],
                       nrels=len(rels_by.get(c, [])),
                       embedding_surface=(names[c] + " | " + (node[c]["description"] or "(no description)"))[:160],
                       props=json.dumps(merged_spec(c), default=str)[:200]) for c, r, sc in ranked[:3]],
        best_rank=(best_rank if best_rank < 9999 else ">512")))
for f in FORENSICS:
    rprint(f"[bold]{f['pid']}[/bold] gold={f['gold']!r} ({f['rule']})  Q: {f['question']}")
    for c in f["carriers"]:
        rprint(f"   carrier {c['name']!r} ({','.join(c['types'])})  sim-rank=[bold]{c['rank']}[/bold] "
               f"score={c['score']} nrels={c['nrels']}")
        rprint(f"     embed-surface: {c['embedding_surface']}")
        rprint(f"     props: {c['props']}")

W004 gold='PS0001464' (catalogue_code)  Q: What is the part number of the Continuous Positive Airway Pressure?

carrier 'F&P Sleepstyle Auto CPAP' (CPAPDevice)  sim-rank=98 score=0.677 nrels=2

embed-surface: F&P Sleepstyle Auto CPAP | (no description)

props: {"max_allowed_qty": "1", "approved_price": "$554.00", "device_code": "PS0001464", "operating_mode": 
"Auto CPAP"}

W010 gold='1097348' (catalogue_code)  Q: What is the part number of the Cannula, Pro-Flow, nasal, pediatric 10 pk?

...

props: {"integration": "integrated", "purpose": "reduce nasal dryness and irritation", "type": "heated", 
"optional": "yes", "function": "humidification and warming of air", "benefit": "reduces nasal dryness

W066 gold='WM31660' (catalogue_code)  Q: What is the part number of the DC Adapter 12/24 V?

carrier 'prisma HUB' (Accessory)  sim-rank=>512 score=None nrels=1

embed-surface: prisma HUB | Modem for prisma CLOUD telemedical connection for plus variants

props: {"order_number": "WM 31660"}

## Task 2 - Policy replay (read-only render-time simulation)

**Policy A** - raise the per-product relation render cap 15 -> 40 on every seed.
**Policy B** - deterministic SKU/accessory->device carrier linkage, simulated at render time: a load-time link map built from three deterministic rules (E: accessory-typed entity with a non-SIMILAR edge into the seed; P: any entity holding a property value equal to the seed's name; N: accessory-typed entity whose full text references a digit-named seed), then linked carriers' canonical renders (+attached propositions) appended to the seed's render. Scoped variants sweep append depth (top-n seeds, per-seed fan-in cap orphan-first, carrier size gate) to find the cheapest sufficient mix.


In [7]:
# ---- Policy A: relation render cap 15 -> 40, re-score the 8 + census-wide token add ----
recA = [pid for pid in EIGHT if det_present(pby[pid]["gold_evidence"][0], TOP64[pid], cap=40)]
addA = sum(max(0, TOK(seed_render(n, 40)) - TOK(seed_render(n, 15))) for p in W for n in TOP64[p["id"]])
rprint(f"[bold]Policy A (rel cap 15->40)[/bold] recovers {len(recA)}/8 {recA}  "
       f"added tokens = {addA} (+{100*addA/BASE_TOK:.2f}%)  -> [red]INERT[/red]" if not recA else f"recovers {recA}")

Policy A (rel cap 15->40) recovers 0/8 []  added tokens = 80095 (+3.74%)  -> INERT

In [8]:
# ---- Policy B: deterministic load-time link maps (E / P / N) ----
ACC_LABELS = {"Accessory", "SKU", "ComfortFeature", "Feature", "Component", "Specification"}
def accish(nid): return bool(set(node[nid]["types"]) & ACC_LABELS)
FT = {}
def ftext(c):
    if c not in FT:
        FT[c] = gnorm2(names[c] + " " + (node[c]["description"] or "") + " " +
                       json.dumps(merged_spec(c), default=str) + " " + " ".join(props_by.get(c, [])))
    return FT[c]
acc = [c for c in node if accish(c)]
adjE = defaultdict(set)                                        # rule E: accessory edge into seed
for c in acc:
    for _, b in rels_by.get(c, []): adjE[b].add(c)
name_norm = {n: gnorm2(names[n]) for n in node}
linkP = defaultdict(set); val_index = defaultdict(set)         # rule P: prop value == seed name
for c in node:
    for v in merged_spec(c).values():
        vn = gnorm2(str(v))
        if len(vn) >= 4: val_index[vn].add(c)
for dd, nn in name_norm.items():
    if len(nn) >= 4 and nn in val_index:
        for c in val_index[nn] - {dd}: linkP[dd].add(c)
digit_seeds = [dd for dd, nn in name_norm.items() if len(nn) >= 6 and any(ch.isdigit() for ch in nn)]
linkN = defaultdict(set)                                       # rule N: digit-named seed referenced in accessory text
for dd in digit_seeds:
    nn = name_norm[dd]
    for c in acc:
        if c != dd and nn in ftext(c): linkN[dd].add(c)
LINK = defaultdict(set)
for m_ in (adjE, linkP, linkN):
    for dd, cs in m_.items(): LINK[dd] |= cs
def ctx_of(c): return seed_render(c, 15) + " " + " ".join(props_by.get(c, []))
TOKC = {}
def tokc(c):
    if c not in TOKC: TOKC[c] = TOK(ctx_of(c))
    return TOKC[c]
def nrels(c): return len(rels_by.get(c, []))
rprint(f"[green]link maps[/green] E targets={len(adjE)} P targets={len(linkP)} N targets={len(linkN)} "
       f"(union targets={len(LINK)}, accessory-ish carriers={len(acc)})")

link maps E targets=1005 P targets=224 N targets=385 (union targets=1266, accessory-ish carriers=1716)

In [9]:
# ---- Policy B evaluator: render-time append simulation (NO graph writes) + census-wide token accounting ----
def evaluate_B(label, n_seeds=64, cap=999, size_gate=None):
    add = 0; hit_set = set(); rec8 = set()
    for p in W:
        t64 = TOP64[p["id"]]; t64s = set(t64); seen = set(); extra = []
        for n in t64[:n_seeds]:
            picked = 0
            for c in sorted(LINK.get(n, ()), key=lambda c: (nrels(c), tokc(c), c)):   # orphan-first deterministic
                if picked >= cap: break
                if c in seen or c in t64s: continue
                if size_gate and tokc(c) > size_gate: continue
                seen.add(c); extra.append(ctx_of(c)); add += tokc(c); picked += 1
        ex = " ".join(extra)
        for g in p["gold_evidence"]:
            if det_present(g, t64, extra=ex):
                hit_set.add((p["id"], g))
                if p["id"] in EIGHT: rec8.add(p["id"])
    pct = 100 * add / BASE_TOK
    regress = BASE_HITS - hit_set
    rprint(f"{label:<44} hits={len(hit_set)} rec8={len(rec8)}/8 {sorted(rec8)} add={add} (+{pct:.2f}%, "
           f"{pct/KNEE_PCT:.1f}x knee) regressions={len(regress)}")
    return dict(label=label, hits=len(hit_set), rec8=sorted(rec8), add_tokens=add, add_pct=round(pct, 2),
                knee_multiple=round(pct / KNEE_PCT, 2), regressions=len(regress), hit_set=hit_set)
SWEEP = [
    evaluate_B("B-blanket (E+P+N, all 64 seeds, uncapped)"),
    evaluate_B("B n=12 cap=2 size<=300", 12, 2, 300),
    evaluate_B("B n=11 cap=2 size<=300", 11, 2, 300),
    evaluate_B("B n=4 cap=2 size<=600 (in-budget candidate)", 4, 2, 600),
    evaluate_B("B n=12 cap=2 size<=250 (knee-edge candidate)", 12, 2, 250),
]

B-blanket (E+P+N, all 64 seeds, uncapped)    hits=82 rec8=5/8 ['W012', 'W044', 'W046', 'W063', 'W065'] add=6178267 
(+288.70%, 51.6x knee) regressions=0

B n=12 cap=2 size<=300                       hits=80 rec8=3/8 ['W012', 'W044', 'W063'] add=141333 (+6.60%, 1.2x 
knee) regressions=0

B n=11 cap=2 size<=300                       hits=80 rec8=3/8 ['W012', 'W044', 'W063'] add=130459 (+6.10%, 1.1x 
knee) regressions=0

B n=4 cap=2 size<=600 (in-budget candidate)  hits=79 rec8=2/8 ['W012', 'W063'] add=64089 (+2.99%, 0.5x knee) 
regressions=0

B n=12 cap=2 size<=250 (knee-edge candidate) hits=79 rec8=2/8 ['W012', 'W063'] add=119065 (+5.56%, 1.0x knee) 
regressions=0

## Task 3 - Zero-regression replay + firing geometry

The scorer is monotone under appended render text (exact_present and word-overlap can only flip miss -> hit), so regressions are structurally impossible - verified empirically anyway: all 77 pinned baseline hits must survive under every policy mix.


In [10]:
# ---- firing geometry: which (seed rank, rule, carrier) scores each recoverable gold ----
def rules_of(c, dseed):
    out = []
    if c in adjE.get(dseed, ()): out.append("E")
    if c in linkP.get(dseed, ()): out.append("P")
    if c in linkN.get(dseed, ()): out.append("N")
    return "/".join(out)
GEOM = {}
for pid in EIGHT:
    p = pby[pid]; gold = p["gold_evidence"][0]; t64 = TOP64[pid]; t64s = set(t64)
    fires = []
    for rk, dseed in enumerate(t64, 1):
        for c in LINK.get(dseed, ()):
            if c in t64s: continue
            if exact_present(gold, ctx_of(c)):
                fires.append(dict(seed_rank=rk, seed=names[dseed], rule=rules_of(c, dseed),
                                  carrier=names[c], nrels=nrels(c), ctx_tok=tokc(c)))
    GEOM[pid] = fires
    rprint(f"{pid}: {len(fires)} firing pairs " + ("; ".join(f"seed#{f['seed_rank']} {f['seed']!r} -{f['rule']}-> {f['carrier']!r} ({f['ctx_tok']} tok)" for f in fires[:3]) if fires else "[red]NONE - unreachable by the linkage family[/red]"))

# ---- zero-regression: every SWEEP entry must retain all 77 baseline hits ----
for svar in SWEEP:
    lost = BASE_HITS - svar["hit_set"]
    assert not lost, f"REGRESSION under {svar['label']}: {lost}"
rprint("[bold green]zero-regression PASS[/bold green] - all 77 pinned hits survive under every policy mix (scorer is monotone under appended text)")

W004: 0 firing pairs NONE - unreachable by the linkage family

W010: 0 firing pairs NONE - unreachable by the linkage family

W012: 1 firing pairs seed#2 'Bi-level' -P-> 'AirCurve 10 S' (192 tok)

W044: 3 firing pairs seed#11 '20 cmH2O' -N-> 'ClimateLineAir 11' (1280 tok); seed#11 '20 cmH2O' -N-> 'Standard air 
tubing' (283 tok); seed#11 '20 cmH2O' -N-> 'SlimLine' (977 tok)

W046: 1 firing pairs seed#19 'AirSense 11' -E/N-> 'Air Filter' (1513 tok)

W063: 2 firing pairs seed#3 'Pro-Flow nasal cannula adult' -E-> 'Pro-Flow sample pack' (152 tok); seed#6 'Pro-Flow 
nasal cannula' -E-> 'Pro-Flow sample pack' (152 tok)

W065: 1 firing pairs seed#2 'RESmart CPAP' -E-> 'Integrated Heated Humidifier' (507 tok)

W066: 0 firing pairs NONE - unreachable by the linkage family

zero-regression PASS - all 77 pinned hits survive under every policy mix (scorer is monotone under appended text)

## Task 4 - Adjudication against the registered bars

Bar: >= 5/8 recovered at unchanged k, within the H171 knee budget (+5.60% census render tokens), zero regression on the 77 pinned hits. Registered refuter: if recovery requires exceeding the knee budget, the residue goes to the gap ledger as priced-out abstention territory (H161 doctrine).


In [11]:
# ---- adjudication + machine-readable report ----
CAUSE = {
 "W004": "rank-position + embedding-surface mismatch: single carrier 'F&P Sleepstyle Auto CPAP' (ADP catalogue-row device, empty description - the index sees only the brand name) ranks 98 against a category-phrased query ('Continuous Positive Airway Pressure'); no linkage rule reaches it (its only neighbours - MoH ADP, Fisher & Paykel - never rank; prop-val target 'Auto CPAP' absent from top-64). UNRECOVERABLE by the registered policy family",
 "W010": "misattributed table proposition (extraction-side): gold 1097348 lives ONLY in a shared markdown part-number table attached to the wrong entities (Alice LoFlo capnography accessories, both >512); their text references no top-64 seed. UNRECOVERABLE - H119 canonicalization territory",
 "W012": "rank-position, linkable: carrier 'AirCurve 10 S' (ADP row, device_code=gold) ranks 100; prop-val linkage (operating_mode 'Bi-level' == top-64 seed#2 'Bi-level') recovers it cheaply",
 "W044": "cross-product spec attachment: best carrier '4-20cm H2O pressure range' (rank 225) belongs to Linde Noctivance, not DreamStation; DreamStation's own 4 cmH2O never extracted onto a DreamStation entity. Recoverable via digit-name linkage at seed#11 '20 cmH2O' - but only above the knee (+6.1%)",
 "W046": "hub fan-in dilution: 17 carriers, best rank 126 ('Universal Power Supply'); the only linked scoring carrier ('Air Filter', 1513 tok) hangs off seed#19 'AirSense 11' behind 83 sibling candidates - reachable only in the blanket tier (+206%)",
 "W063": "orphan accessory, linkable: single carrier 'Pro-Flow sample pack' (>512; code only in props.contents, embedding surface says nothing about adult cannulas); edge linkage to seed#3 'Pro-Flow nasal cannula adult' recovers it cheaply",
 "W065": "misattached spec proposition + fan-in dilution: the '3.5 lbs' weight proposition hangs on 'Integrated Heated Humidifier' (>512), not on RESmart CPAP; edge to seed#2 exists but the carrier sits deep among 51 siblings and needs its propositions rendered - blanket tier only",
 "W066": "misattributed code (extraction variance): 'WM 31660' is stored as prisma HUB's order number (>512, references nothing in top-64) while the probe targets 'DC Adapter 12/24 V' (top-1 seed, which lacks the code). UNRECOVERABLE - H119 territory",
}
best_in_budget = max((s for s in SWEEP if s["add_pct"] <= KNEE_PCT), key=lambda s: (len(s["rec8"]), -s["add_tokens"]))
ceiling = max(SWEEP, key=lambda s: (len(s["rec8"]), -s["add_tokens"]))
bar_met = len(best_in_budget["rec8"]) >= 5
verdict = ("CONFIRMED" if bar_met else
           "REFUTED on clause (b) - recovery ceiling 5/8 exists but only at "
           f"{ceiling['knee_multiple']}x the knee budget; within budget the mix recovers {len(best_in_budget['rec8'])}/8. "
           "The registered refuter fires: the residue is priced-out abstention territory (gap ledger). Clause (a) CONFIRMED - all 8 causes enumerated")
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report = dict(
    hypothesis="R19-H211", utc=stamp, graph="neo4j2", read_only=True, cpu_only=True,
    graph_fingerprint=GRAPH_FP, render_fingerprint=RENDER_FP,
    pinned_baseline=dict(hits=len(BASE_HITS), total=len(wgolds), recall=round(len(BASE_HITS)/len(wgolds), 3),
                         census_tokens=BASE_TOK, knee_pct=KNEE_PCT, knee_budget_tokens=KNEE_BUDGET),
    forensics=[dict(**{k: v for k, v in f.items() if k != "carriers"},
                    carriers=f["carriers"], cause=CAUSE[f["pid"]]) for f in FORENSICS],
    firing_geometry={pid: g for pid, g in GEOM.items()},
    policy_A=dict(recovered=recA, added_tokens=addA, add_pct=round(100*addA/BASE_TOK, 2), verdict="INERT - 0/8"),
    policy_B_sweep=[{k: v for k, v in s.items() if k != "hit_set"} for s in SWEEP],
    frontier=dict(in_budget_best=dict(label=best_in_budget["label"], rec8=best_in_budget["rec8"],
                                      add_pct=best_in_budget["add_pct"]),
                  ceiling=dict(label=ceiling["label"], rec8=ceiling["rec8"], add_pct=ceiling["add_pct"],
                               knee_multiple=ceiling["knee_multiple"])),
    zero_regression=dict(baseline_hits=77, survived=True,
                         note="scorer monotone under appended render text; verified for every sweep entry"),
    acceptance=dict(bar=">=5/8 within +5.6% at zero regression", met=bool(bar_met)),
    verdict=verdict)
outp = ROOT / f"reports/unranked-residue-h211-{stamp}.json"
outp.write_text(json.dumps(report, indent=2, default=str))
driver2.close()
rprint(f"[green]report written[/green] {outp}")
rprint(f"[bold]===== R19-H211 VERDICT: {'CONFIRMED' if bar_met else 'PARTIAL - clause (a) CONFIRMED, clause (b) REFUTED (refuter fires: priced-out residue)'} =====[/bold]")

report written ../reports/unranked-residue-h211-20260707T190724Z.json

===== R19-H211 VERDICT: PARTIAL - clause (a) CONFIRMED, clause (b) REFUTED (refuter fires: priced-out residue) 
=====